## 00. Introduction + requirements
## 00. Giới thiệu + yêu cầu

**English**

This notebook is the public, production-style Kaggle NVIDIA T4 x2 demo for
**Mage-Flow-Turbo** (text-to-image) and **Mage-Flow-Edit-Turbo** (instruction-based
image editing, also called "Edit" here). The target runtime is:

| Component | Device | Internal worker |
| :--- | :--- | :--- |
| Mage-Flow-Turbo (T2I) | `cuda:0` | `127.0.0.1:8101` |
| Mage-Flow-Edit-Turbo (Edit) | `cuda:1` | `127.0.0.1:8102` |
| REST coordinator | CPU | `http://127.0.0.1:8090` |

This is a **temporary Kaggle session, production-style demo** only: it is not
SLA-backed, not highly available, not multi-tenant, and it has **no CPU
fallback**. The notebook drives the project's own scripts and REST API; it does
not re-implement the model server.

**Tiếng Việt**

Notebook này là bản demo public theo phong cách production trên **Kaggle
NVIDIA T4 x2** dành cho **Mage-Flow-Turbo** (text-to-image) và
**Mage-Flow-Edit-Turbo** (chỉnh sửa ảnh theo câu lệnh, gọi tắt "Edit"). Bố cục
runtime mục tiêu:

| Thành phần | Thiết bị | Worker nội bộ |
| :--- | :--- | :--- |
| Mage-Flow-Turbo (T2I) | `cuda:0` | `127.0.0.1:8101` |
| Mage-Flow-Edit-Turbo (Edit) | `cuda:1` | `127.0.0.1:8102` |
| REST coordinator | CPU | `http://127.0.0.1:8090` |

Đây chỉ là **session Kaggle tạm thời, bản demo theo phong cách production**:
không có SLA, không HA, không multi-tenant và **không có CPU fallback**.
Notebook điều phối script và REST API của chính dự án, không sao chép lại
server model.

## Requirements / Yêu cầu

- Kaggle accelerator: **NVIDIA T4 x2** (enabled once).
- Internet: **ON** — Stage 01 fetches the exact public project source from GitHub.
- Three read-only Kaggle inputs attached:
  - `dangkhoa2016/mage-flow-community-mage-flow-turbo` (T2I model).
  - `dangkhoa2016/mage-flow-community-mage-flow-edit-turbo` (Edit model).
  - `dangkhoa2016/mage-flow-t4x2-runtime-cache` (canonical Mage runtime archive).
- Stage 01 verifies the canonical runtime archive, restores or reuses the runtime cache, materializes the exact public GitHub source at the pinned SHA, verifies runtime provenance, and only then imports project modules.
- `RUNTIME_ROOT` is the runtime reuse guard: bootstrap restore count is 0 when already present and 1 when restored in this session.
- No terminal preparation is required.


## 01. Configuration
## 01. Cấu hình

**English**

Define public paths, REST port, fixed device routing, the per-session API token,
and reusable notebook helpers (subprocess runner with heartbeat, REST headers,
image helpers). Stage 01 first verifies the attached canonical runtime archive,
restores or reuses the runtime cache, then materializes the exact public GitHub
project source at the pinned immutable commit, verifies runtime provenance, and
only then imports project modules. The
token is read from the project `.runtime/api_token` file when it exists so the
notebook always uses the same temporary Bearer token as the coordinator; it is
**never printed**.

**Tiếng Việt**

Khai báo đường dẫn public, REST port, ánh xạ GPU cố định, token API theo
session và các hàm tiện ích tái sử dụng của notebook (chạy lệnh con kèm
heartbeat, header REST, tiện ích ảnh). Stage 01 trước tiên xác minh archive runtime
chuẩn đã attach, khôi phục hoặc tái sử dụng runtime cache, sau đó đưa đúng mã
nguồn public từ GitHub tại commit bất biến đã chốt vào `PROJECT_ROOT`, xác minh
provenance runtime, và chỉ sau đó mới import các module của dự án. Token được
đọc từ file `.runtime/api_token` của dự án nếu đã tồn
tại để notebook luôn dùng đúng Bearer token tạm giống coordinator; token
**không bao giờ được in ra**.


In [ ]:
from pathlib import Path
import base64, hashlib, io, json, os, secrets, shutil, subprocess, sys, threading, time
import urllib.request

PROJECT_REPO_URL = "https://github.com/dangkhoa2016/Mage-Flow-Turbo-Dual-T4-REST-API.git"
EXPECTED_PROJECT_SOURCE_SHA = "db6e3fba417b2f6aa0dc5e1dbaf1ccae68c892e0"
PROJECT_ROOT = Path("/kaggle/working/mage-flow-t4x2-production-rest-api-demo").resolve()
RUNTIME_INPUT_ROOT = Path(
    "/kaggle/input/datasets/dangkhoa2016/mage-flow-t4x2-runtime-cache"
)
RUNTIME_ARCHIVE = RUNTIME_INPUT_ROOT / "mage-flow-t4x2-c1-runtime-py312-torch213-cu126.tar.zst"
EXPECTED_RUNTIME_ARCHIVE_SIZE = 3381276345
EXPECTED_RUNTIME_ARCHIVE_SHA256 = "f1cd0174c7f8b508feafd1132bf934aeb8f15e5f7832d7dc0b68d7ac788d62d5"
RUNTIME_ROOT = Path("/kaggle/working/mage-flow-v5-t4x2-c1-concurrency-source-20260912").resolve()
RUNTIME_PYTHON = RUNTIME_ROOT / ".venv" / "bin" / "python"
MAGE_SOURCE = RUNTIME_ROOT / "vendor" / "Mage"
EXPECTED_MAGE_COMMIT = "76bec2bb3818863f470de7e867c2dc7f1d0bfd83"

def sha256_file(path: str) -> str:
    h = hashlib.sha256()
    with open(path, "rb") as fh:
        for chunk in iter(lambda: fh.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

def require_file(path: str, label: str) -> Path:
    p = Path(path)
    if not p.is_file():
        raise RuntimeError(f"{label} missing: {p}")
    return p

def verify_exact_size(path: str, expected: int, label: str) -> None:
    size = Path(path).stat().st_size
    if not (size == expected):
        raise RuntimeError(f"{label} size mismatch: expected {expected} found {size}")
    print(f"[PASS] {label} size={size}")

def verify_exact_sha256(path: str, expected: str, label: str) -> None:
    actual = sha256_file(path)
    if not (actual == expected):
        raise RuntimeError(f"{label} sha256 mismatch: expected {expected} found {actual}")
    print(f"[PASS] {label} sha256={actual}")

def git_head(repo: str) -> str:
    out = subprocess.run(["git", "-C", str(repo), "rev-parse", "HEAD"], capture_output=True, text=True)
    if not (out.returncode == 0):
        raise RuntimeError(f"git rev-parse HEAD failed in {repo}")
    return out.stdout.strip()

def restore_project_source_once() -> int:
    if PROJECT_ROOT.is_dir():
        head = git_head(PROJECT_ROOT)
        if not (head == EXPECTED_PROJECT_SOURCE_SHA):
            raise RuntimeError(f"existing PROJECT_ROOT HEAD {head} != {EXPECTED_PROJECT_SOURCE_SHA}")
        print("[INFO] PROJECT_SOURCE_BOOTSTRAP_COUNT=0 exact-SHA source reused")
        return 0
    tmp = Path("/kaggle/working/project-source-bootstrap-tmp")
    if tmp.is_dir():
        shutil.rmtree(tmp)
    subprocess.run(["git", "clone", "--no-checkout", PROJECT_REPO_URL, str(tmp)], check=True, capture_output=True, text=True)
    subprocess.run(["git", "-C", str(tmp), "checkout", "--detach", EXPECTED_PROJECT_SOURCE_SHA], check=True, capture_output=True, text=True)
    head = git_head(tmp)
    if not (head == EXPECTED_PROJECT_SOURCE_SHA):
        raise RuntimeError(f"clone HEAD {head} != {EXPECTED_PROJECT_SOURCE_SHA}")
    os.rename(str(tmp), str(PROJECT_ROOT))
    print("[INFO] PROJECT_SOURCE_BOOTSTRAP_COUNT=1 exact-SHA source materialized")
    return 1

def restore_runtime_once() -> int:
    if RUNTIME_ROOT.is_dir():
        print("[INFO] BOOTSTRAP_RUNTIME_RESTORE_COUNT=0 runtime already restored")
        return 0
    subprocess.run(["tar", "-I", "zstd", "-xpf", str(RUNTIME_ARCHIVE), "-C", "/kaggle/working"], check=True, capture_output=True, text=True)
    print("[INFO] BOOTSTRAP_RUNTIME_RESTORE_COUNT=1 runtime restored exactly once")
    return 1

print("[INFO] project_source_transport=public_github_exact_sha")
for tool in ("tar", "zstd", "git"):
    if shutil.which(tool) is None:
        raise RuntimeError(f"required tool missing: {tool}")
require_file(RUNTIME_ARCHIVE, "runtime archive")
verify_exact_size(RUNTIME_ARCHIVE, EXPECTED_RUNTIME_ARCHIVE_SIZE, "runtime archive")
verify_exact_sha256(RUNTIME_ARCHIVE, EXPECTED_RUNTIME_ARCHIVE_SHA256, "runtime archive")
BOOTSTRAP_RUNTIME_RESTORE_COUNT = restore_runtime_once()
PROJECT_SOURCE_BOOTSTRAP_COUNT = restore_project_source_once()
print(f"[INFO] PROJECT_SOURCE_BOOTSTRAP_COUNT={PROJECT_SOURCE_BOOTSTRAP_COUNT}")
print(f"[INFO] BOOTSTRAP_RUNTIME_RESTORE_COUNT={BOOTSTRAP_RUNTIME_RESTORE_COUNT}")
require_file(RUNTIME_PYTHON, "runtime Python")
if not (MAGE_SOURCE / ".git").is_dir():
    raise RuntimeError("Mage .git missing after runtime restore")
mage_head = git_head(MAGE_SOURCE)
print(f"[INFO] mage_commit={mage_head}")
if not (mage_head == EXPECTED_MAGE_COMMIT):
    raise RuntimeError(f"expected Mage commit {EXPECTED_MAGE_COMMIT} found {mage_head}")
for required_file in ("pyproject.toml", "scripts/token_store.py", "scripts/run_public_candidate.sh", "scripts/acceptance.py", "scripts/validate_vendor_source.py", "server/app.py"):
    require_file(PROJECT_ROOT / required_file, f"project source {required_file}")
sys.path.insert(0, str(PROJECT_ROOT))
from scripts.token_store import ensure_token

T2I_MODEL = Path("/kaggle/input/models/dangkhoa2016/mage-flow-community-mage-flow-turbo/pytorch/default/1")
EDIT_MODEL = Path("/kaggle/input/models/dangkhoa2016/mage-flow-community-mage-flow-edit-turbo/pytorch/default/1")
EDIT_SOURCE = EDIT_MODEL / "assets" / "dog.jpg"
REST_HOST = "127.0.0.1"
REST_PORT = 8090
REST_URL = f"http://{REST_HOST}:{REST_PORT}"
T2I_DEVICE = "cuda:0"
EDIT_DEVICE = "cuda:1"
OUTPUT_DIR = PROJECT_ROOT / "outputs"
ACCEPT_DIR = PROJECT_ROOT / ".runtime" / "acceptance"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
ACCEPT_DIR.mkdir(parents=True, exist_ok=True)
BAR = "━" * 46

def stage(label: str) -> None:
    print(BAR); print(f"[STAGE] {label}"); print(BAR)

def heartbeat(msg: str) -> None:
    print(f"[HEARTBEAT] {msg}", flush=True)

def run_cmd(cmd, label: str, heartbeat_seconds: int = 45) -> int:
    stage(label)
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                            text=True, bufsize=1)
    stop_hb = threading.Event()
    started = time.monotonic()
    def _hb() -> None:
        while not stop_hb.wait(heartbeat_seconds):
            heartbeat(f"{label} still running elapsed={int(time.monotonic() - started)}s")
    hb = threading.Thread(target=_hb, daemon=True)
    hb.start()
    try:
        try:
            for line in proc.stdout:
                print(line, end="", flush=True)
        except Exception as exc:
            print(f"[WARN] output stream error: {type(exc).__name__}: {exc}", flush=True)
        proc.wait()
    finally:
        stop_hb.set()
        if proc.stdout is not None:
            proc.stdout.close()
    if proc.returncode != 0:
        raise RuntimeError(f"command failed rc={proc.returncode}: {' '.join(map(str, cmd))}")
    print(f"[PASS] {label} rc=0")
    return 0

def resolve_token():
    token_file = PROJECT_ROOT / ".runtime" / "api_token"
    from scripts.token_store import ensure_token as _ensure_token
    token = _ensure_token(token_file)
    os.environ["MAGE_FLOW_API_TOKEN"] = token
    return token

TOKEN = resolve_token()

def api_headers() -> dict:
    return {"Authorization": f"Bearer {TOKEN}"}

def sha256_hex(raw: bytes) -> str:
    return hashlib.sha256(raw).hexdigest()

def decode_data_url(data_url: str) -> bytes:
    _head, _, b64 = data_url.partition(",")
    return base64.b64decode(b64)

print(f"[INFO] PROJECT_ROOT={PROJECT_ROOT}")
print(f"[INFO] REST={REST_URL}")
print(f"[INFO] T2I_DEVICE={T2I_DEVICE} EDIT_DEVICE={EDIT_DEVICE}")
print("[INFO] MAGE_FLOW_API_TOKEN=<temporary session token; not printed>")
if not (T2I_DEVICE == "cuda:0" and EDIT_DEVICE == "cuda:1"):
    raise RuntimeError("device configuration mismatch: cuda:0/cuda:1 required")
print("[PASS] NOTEBOOK_CONFIGURATION")


## 02. Kaggle environment preflight
## 02. Kiểm tra môi trường Kaggle

**English**

Inspect Python, platform, working/input paths, and disk usage before touching
the models. Expected output is environment + storage information. If this cell
fails, the Kaggle session needs to be recreated.

**Tiếng Việt**

Kiểm tra Python, nền tảng, đường dẫn working/input và dung lượng đĩa trước khi
đụng tới model. Output mong đợi là thông tin môi trường + storage. Nếu cell
này lỗi, cần tạo lại session Kaggle.


In [ ]:
import platform, shutil, sys
stage("Kaggle environment preflight")
print(f"[INFO] python={sys.version.split()[0]}")
print(f"[INFO] platform={platform.platform()}")
for p in ("/kaggle/input", "/kaggle/working"):
    print(f"[INFO] exists {p}={Path(p).exists()}")
usage = shutil.disk_usage("/kaggle/working")
print(f"[INFO] working_disk_gb used={usage.used/2**30:.2f} free={usage.free/2**30:.2f}")
if not (Path("/kaggle/working").exists()):
    raise RuntimeError('Path("/kaggle/working").exists()')
print("[PASS] KAGGLE_ENVIRONMENT_PREFLIGHT")


## 03. Verify NVIDIA T4 x2
## 03. Xác minh NVIDIA T4 x2

**English**

Require exactly the Kaggle T4 x2 topology: at least two CUDA devices whose names
contain `T4`, with the fixed split-device contract (`cuda:0` T2I, `cuda:1`
Edit). There is **no CPU fallback**: if CUDA is absent or the topology differs,
this cell must fail and the session must not proceed.

**Tiếng Việt**

Bắt buộc đúng topo T4 x2 của Kaggle: ít nhất hai thiết bị CUDA có tên chứa
`T4`, với contract chia GPU cố định (`cuda:0` cho T2I, `cuda:1` cho Edit).
**Không có CPU fallback**: nếu thiếu CUDA hoặc topo khác, cell này phải fail và
session không được tiếp tục.


In [ ]:
stage("Verify NVIDIA T4 x2")
import torch
if not torch.cuda.is_available():
    raise RuntimeError("CUDA unavailable; enable Kaggle T4 x2 once")
count = torch.cuda.device_count()
print(f"[INFO] CUDA devices detected: {count}")
if not count >= 2:
    raise RuntimeError(f"expected >=2 CUDA devices, found {count}")
names = [torch.cuda.get_device_name(i) for i in range(count)]
for i in range(count):
    print(f"[INFO] cuda:{i} = {names[i]}")
if not ("T4" in names[0].upper() and "T4" in names[1].upper()):
    raise RuntimeError(f"expected NVIDIA T4 x2, got {names[:2]}")
if not (T2I_DEVICE == "cuda:0" and EDIT_DEVICE == "cuda:1"):
    raise RuntimeError("device configuration mismatch: cuda:0/cuda:1 required")
print("[PASS] T4_X2_HARDWARE_GATE cuda:0=T4 cuda:1=T4")


## 04. Input / model discovery
## 04. Khám phá model input

**English**

Locate the two read-only Kaggle Model inputs by their expected public
identities. Expected output prints both resolved model roots and the Edit sample
image. Failure means the required Kaggle Models are not attached. We only verify
existence here — **no weights are loaded yet**.

**Tiếng Việt**

Tìm hai Kaggle Model input read-only theo đúng định danh public. Output mong đợi
in cả hai model root và ảnh mẫu dùng cho Edit. Nếu lỗi nghĩa là các Kaggle
Model bắt buộc chưa được attach. Ở bước này chỉ kiểm tra tồn tại — **chưa load
bất kỳ weight nào**.


In [ ]:
stage("Input / model discovery")
for label, p in (("T2I_MODEL", T2I_MODEL), ("EDIT_MODEL", EDIT_MODEL)):
    print(f"[INFO] {label}={p}")
    if not (p.is_dir()):
        raise RuntimeError(f"{label} not attached: {p}")
print(f"[INFO] EDIT_SOURCE_IMAGE={EDIT_SOURCE}")
if not (EDIT_SOURCE.is_file()):
    raise RuntimeError("Edit sample source image missing")
print("[PASS] MODEL_INPUT_DISCOVERY")


## 05. Runtime validation / reuse
## 05. Xác minh / tái sử dụng runtime

**English**

Verify the restored canonical Mage runtime matches the exact pinned provenance.
A genuinely fresh session restores the already-qualified immutable runtime cache
exactly once from the attached `dangkhoa2016/mage-flow-t4x2-runtime-cache` Dataset. After
bootstrap there is no second extraction, no rebuild, no re-download and no
dependency reinstall. Outputs: runtime input Dataset, archive identity, runtime
Python path, committed Mage HEAD and the bootstrap restore count. A mismatch
means the restored runtime is not the canonical one and the session must stop.

**Tiếng Việt**

Xác minh runtime Mage chuẩn được khôi phục khớp đúng provenance đã chốt. Một
session mới thực sự sẽ khôi phục runtime cache bất biến đã được kiểm định đúng
một lần từ Dataset `dangkhoa2016/mage-flow-t4x2-runtime-cache` được attach. Sau bootstrap
không có lần giải nén, rebuild, tải lại hay cài đặt lại dependency nào. Output
gồm: Dataset runtime input, định danh archive, đường dẫn Python runtime, commit
Mage hiện tại và số lần khôi phục bootstrap. Sai lệch nghĩa là runtime đã khôi
phục không phải bản chuẩn và session phải dừng.


In [ ]:
stage("Runtime validation / reuse")
print(f"[INFO] runtime_input_dataset=dangkhoa2016/mage-flow-t4x2-runtime-cache")
print(f"[INFO] runtime_archive_filename={RUNTIME_ARCHIVE.name}")
print(f"[INFO] runtime_archive_size={EXPECTED_RUNTIME_ARCHIVE_SIZE}")
print(f"[INFO] runtime_archive_sha256={EXPECTED_RUNTIME_ARCHIVE_SHA256}")
print(f"[INFO] runtime_root={RUNTIME_ROOT}")
print(f"[INFO] runtime_python={RUNTIME_PYTHON}")
print(f"[INFO] BOOTSTRAP_RUNTIME_RESTORE_COUNT={BOOTSTRAP_RUNTIME_RESTORE_COUNT}")
if not (RUNTIME_PYTHON.is_file()):
    raise RuntimeError(f"validated runtime Python missing: {RUNTIME_PYTHON}")
if not (MAGE_SOURCE.is_dir()):
    raise RuntimeError(f"Mage source missing: {MAGE_SOURCE}")
actual = subprocess.run(["git", "-C", str(MAGE_SOURCE), "rev-parse", "HEAD"],
                        capture_output=True, text=True).stdout.strip()
print(f"[INFO] mage_commit={actual}")
if not (actual == EXPECTED_MAGE_COMMIT):
    raise RuntimeError(f"commit mismatch: expected {EXPECTED_MAGE_COMMIT} found {actual}")
identity = subprocess.run([str(RUNTIME_PYTHON), "-c",
    "import sys; print(sys.version.split()[0])"], capture_output=True, text=True)
print(f"[INFO] runtime python version: {identity.stdout.strip()}")
if not (identity.returncode == 0):
    raise RuntimeError('identity.returncode == 0')
if not (BOOTSTRAP_RUNTIME_RESTORE_COUNT in (0, 1)):
    raise RuntimeError("BOOTSTRAP_RUNTIME_RESTORE_COUNT must be 0 or 1")
print("[INFO] runtime restored exactly once from the declared immutable Dataset; no re-extraction, rebuild, re-download, or dependency reinstall")
print("[PASS] RUNTIME_VALIDATION_REUSE")


## 06. Split-device configuration
## 06. Cấu hình chia GPU cố định

**English**

Create the immutable public routing configuration used by both workers and the
coordinator. Expected output confirms `T2I=cuda:0`, `Edit=cuda:1` and
`cpu_fallback=false`. Any other mapping must fail.

**Tiếng Việt**

Tạo cấu hình routing public bất biến được cả hai worker và coordinator dùng.
Output mong đợi xác nhận `T2I=cuda:0`, `Edit=cuda:1` và `cpu_fallback=false`.
Mọi mapping khác phải fail.


In [ ]:
stage("Split-device configuration")
routing = {
    "t2i_device": T2I_DEVICE,
    "edit_device": EDIT_DEVICE,
    "cpu_fallback": False,
    "coordinator": REST_URL,
}
print(json.dumps(routing, indent=2))
if not (routing == {
    "t2i_device": "cuda:0",
    "edit_device": "cuda:1",
    "cpu_fallback": False,
    "coordinator": REST_URL,
}):
    raise RuntimeError('routing == {\n    "t2i_device": "cuda:0",\n    "edit_device": "cuda:1",\n    "cpu_fallback": False,\n    "coordinator": REST_URL,\n}')
print("[PASS] SPLIT_DEVICE_CONFIGURATION")


## 07. Start both resident workers (orchestrator)
## 07. Khởi động cả hai worker thường trú (orchestrator)

**English**

Call the project's single idempotent orchestrator
(`scripts/run_public_candidate.sh`). On a fresh session it validates the T4 x2
gate, loads T2I (`cuda:0`) and Edit (`cuda:1`) **once each**, starts the
coordinator, and runs the full dual-worker REST acceptance while keeping both
model workers resident. On a later invocation it **reuses** the healthy resident
workers instead of reloading them. Model load emits long-running heartbeats.

**Tiếng Việt**

Gọi orchestrator idempotent duy nhất của dự án (`scripts/run_public_candidate.sh`).
Với session mới, script này kiểm tra cổng T4 x2, load T2I (`cuda:0`) và Edit
(`cuda:1`) **mỗi model một lần**, khởi động coordinator và chạy toàn bộ
acceptance REST hai worker, giữ cả hai model worker thường trú. Lần gọi sau nó
sẽ **tái sử dụng** worker đang khỏe mạnh thay vì load lại. Quá trình load model
phát heartbeat theo thời gian dài.


In [ ]:
run_cmd(["bash", str(PROJECT_ROOT / "scripts" / "run_public_candidate.sh")],
         "Start/reuse both resident workers + coordinator + live acceptance",
         heartbeat_seconds=45)


## 08. Start / verify coordinator
## 08. Khởi động / xác minh coordinator

**English**

The orchestrator already starts the coordinator on `127.0.0.1:8090`. This cell
verifies coordinator liveness via the public `GET /health` endpoint and confirms
the process is resident.

**Tiếng Việt**

Orchestrator đã khởi động coordinator trên `127.0.0.1:8090`. Cell này xác minh
trạng thái sống của coordinator qua `GET /health` public và xác nhận tiến trình
đang thường trú.


In [ ]:
import requests
stage("Verify coordinator")
r = requests.get(f"{REST_URL}/health", timeout=10)
print(f"[INFO] GET /health -> {r.status_code} {r.json()}")
if not (r.status_code == 200 and r.json().get("status") == "ok"):
    raise RuntimeError('r.status_code == 200 and r.json().get("status") == "ok"')
server_pid = PROJECT_ROOT / ".runtime" / "server.pid"
pid = server_pid.read_text().strip() if server_pid.is_file() else "none"
print(f"[INFO] coordinator pid={pid}")
print("[PASS] COORDINATOR_VERIFIED")


## 09. Wait for /ready
## 09. Chờ /ready

**English**

Poll the authenticated `GET /ready` endpoint with heartbeat output. `/ready`
becomes `true` **only** when the coordinator, the T2I worker and the Edit worker
are all ready on their required devices. A bounded wait prevents endless loops.

**Tiếng Việt**

Poll endpoint `GET /ready` (có xác thực) kèm heartbeat. `/ready` chỉ trở thành
`true` khi **tất cả** coordinator, T2I worker và Edit worker sẵn sàng trên đúng
thiết bị yêu cầu. Giới hạn thời gian chờ tránh vòng lặp vô tận.


In [ ]:
stage("Wait for /ready")
import time as _time
wait_started = _time.monotonic()
deadline = wait_started + 300
last_report = 0.0
last_payload = {}
while _time.monotonic() < deadline:
    r = requests.get(f"{REST_URL}/ready", headers=api_headers(), timeout=10)
    last_payload = r.json()
    if r.status_code == 200 and last_payload.get("ready") is True:
        break
    now = _time.monotonic()
    if now - last_report >= 15:
        heartbeat(f"waiting /ready elapsed={int(now - wait_started)}s status={last_payload.get('status')}")
        last_report = now
    _time.sleep(3)
else:
    raise RuntimeError(f"/ready did not become true: {last_payload}")
print(f"[INFO] /ready -> {json.dumps(last_payload, indent=2)}")
if not (last_payload.get("t2i_ready") is True):
    raise RuntimeError('last_payload.get("t2i_ready") is True')
if not (last_payload.get("edit_ready") is True):
    raise RuntimeError('last_payload.get("edit_ready") is True')
print("[PASS] REST_API_READY")


## 10. Inspect /v1/info
## 10. Kiểm tra /v1/info

**English**

Inspect the public runtime identity and device routing returned by the
coordinator. The response must use public model/device names, report
`cpu_fallback=false` and must not expose internal qualification labels.

**Tiếng Việt**

Kiểm tra runtime identity và device routing public do coordinator trả về.
Response phải dùng tên model/device public, báo `cpu_fallback=false` và không
lộ nhãn qualification nội bộ.


In [ ]:
stage("Inspect /v1/info")
r = requests.get(f"{REST_URL}/v1/info", headers=api_headers(), timeout=10)
if not (r.status_code == 200):
    raise RuntimeError(r.text)
info = r.json()
print(json.dumps(info, indent=2))
t2i, edit = info["t2i"], info["edit"]
if not (t2i["device"] == "cuda:0" and t2i["model"] == "mage-flow-turbo" and t2i["ready"] is True):
    raise RuntimeError('t2i["device"] == "cuda:0" and t2i["model"] == "mage-flow-turbo" and t2i["ready"] is True')
if not (edit["device"] == "cuda:1" and edit["model"] == "mage-flow-edit-turbo" and edit["ready"] is True):
    raise RuntimeError('edit["device"] == "cuda:1" and edit["model"] == "mage-flow-edit-turbo" and edit["ready"] is True')
if not (info["cpu_fallback"] is False):
    raise RuntimeError('info["cpu_fallback"] is False')
print("[PASS] INFO_ENDPOINT_OK")


## 11. T2I REST request
## 11. Gửi request T2I qua REST

**English**

Send a real, deterministic text-to-image request to the authenticated
`POST /v1/images/generations` endpoint while both model workers stay resident.
Record prompt, seed, steps, resolution, device, latency and the output SHA-256.
The safe public test request uses a tranquil lake scene at 1024x1024.

**Tiếng Việt**

Gửi một request text-to-image thật, có tính xác định đến endpoint có xác thực
`POST /v1/images/generations` trong khi cả hai model worker vẫn thường trú.
Ghi lại prompt, seed, steps, resolution, device, latency và SHA-256 output.
Request test public an toàn dùng cảnh hồ núi yên bình ở 1024x1024.


In [ ]:
stage("T2I REST request")
import time as _time
T2I_PROMPT = "A tranquil mountain lake at sunrise, photorealistic landscape photography"
t2i_payload = {"prompt": T2I_PROMPT, "seed": 42, "steps": 4, "width": 1024, "height": 1024}
_started = _time.monotonic()
r = requests.post(f"{REST_URL}/v1/images/generations", headers=api_headers(),
                  json=t2i_payload, timeout=600)
t2i_wall = _time.monotonic() - _started
print(f"[INFO] POST /v1/images/generations -> {r.status_code} in {t2i_wall:.2f}s wall")
if not (r.status_code == 200):
    raise RuntimeError(r.text[:500])
t2i = r.json()
if not (t2i["status"] == "completed" and t2i["model"] == "mage-flow-turbo" and t2i["device"] == "cuda:0"):
    raise RuntimeError('t2i["status"] == "completed" and t2i["model"] == "mage-flow-turbo" and t2i["device"] == "cuda:0"')
t2i_raw = decode_data_url(t2i["output"])
t2i_sha = sha256_hex(t2i_raw)
t2i_png = OUTPUT_DIR / "t2i-notebook.png"
t2i_png.write_bytes(t2i_raw)
print(f"[INFO] seed={t2i['seed']} steps={t2i_payload['steps']} size={t2i_payload['width']}x{t2i_payload['height']}")
print(f"[INFO] device={t2i['device']} latency={t2i['elapsed_seconds']}s")
print(f"[INFO] output bytes={len(t2i_raw)} sha256={t2i_sha}")
print(f"[INFO] saved={t2i_png}")
print("[PASS] T2I_REST_REQUEST")


## 12. Display generated T2I image
## 12. Hiển thị ảnh T2I

**English**

Display the actual generated image inline (PIL verified) together with key
request metadata. A textual PASS without the image is not sufficient.

**Tiếng Việt**

Hiển thị trực tiếp ảnh đã tạo (đã xác minh bằng PIL) cùng metadata chính của
request. Chỉ in PASS mà không hiển thị ảnh là chưa đủ.


In [ ]:
from PIL import Image
from IPython.display import Image as IPImage, display
stage("Display generated T2I image")
pil_t2i = Image.open(io.BytesIO(t2i_raw))
if not (pil_t2i.format == "PNG" and pil_t2i.size == (1024, 1024)):
    raise RuntimeError((pil_t2i.format, pil_t2i.size))
display(IPImage(data=t2i_raw, format="png", width=768))
print(f"[INFO] verified PNG {pil_t2i.size[0]}x{pil_t2i.size[1]} sha256={t2i_sha[:16]}...")
print("[PASS] T2I_IMAGE_DISPLAYED")


## 13. Edit REST request
## 13. Gửi request Edit qua REST

**English**

Send a real multipart image-edit request to the authenticated
`POST /v1/images/edits` endpoint using the small safe `dog.jpg` sample image from
the mounted Edit model assets. Record source image, prompt, seed, device,
latency and output SHA-256. No model worker is restarted.

**Tiếng Việt**

Gửi request chỉnh sửa ảnh multipart thật đến endpoint có xác thực
`POST /v1/images/edits`, dùng ảnh mẫu `dog.jpg` an toàn từ assets của Edit model
đã mount. Ghi lại ảnh nguồn, prompt, seed, device, latency và SHA-256 output.
Không worker nào bị khởi động lại.


In [ ]:
stage("Edit REST request")
from PIL import Image
import time as _time
EDIT_PROMPT = "Change the background to a sunny green meadow while keeping the main subject unchanged."
edit_source_bytes = EDIT_SOURCE.read_bytes()
print(f"[INFO] source={EDIT_SOURCE.name} bytes={len(edit_source_bytes)}")
_started = _time.monotonic()
r = requests.post(f"{REST_URL}/v1/images/edits", headers=api_headers(),
                  files={"image": (EDIT_SOURCE.name, edit_source_bytes, "image/jpeg")},
                  data={"prompt": EDIT_PROMPT, "seed": "42"}, timeout=600)
edit_wall = _time.monotonic() - _started
print(f"[INFO] POST /v1/images/edits -> {r.status_code} in {edit_wall:.2f}s wall")
if not (r.status_code == 200):
    raise RuntimeError(r.text[:500])
edit = r.json()
if not (edit["status"] == "completed" and edit["model"] == "mage-flow-edit-turbo" and edit["device"] == "cuda:1"):
    raise RuntimeError('edit["status"] == "completed" and edit["model"] == "mage-flow-edit-turbo" and edit["device"] == "cuda:1"')
edit_raw = decode_data_url(edit["output"])
edit_sha = sha256_hex(edit_raw)
_source_pil = Image.open(io.BytesIO(edit_source_bytes)).convert("RGB")
_src_buf = io.BytesIO(); _source_pil.save(_src_buf, format="PNG")
source_png_raw = _src_buf.getvalue()
(OUTPUT_DIR / "edit-source.png").write_bytes(source_png_raw)
(OUTPUT_DIR / "edit-notebook.png").write_bytes(edit_raw)
print(f"[INFO] prompt={EDIT_PROMPT}")
print(f"[INFO] seed={edit['seed']} device={edit['device']} latency={edit['elapsed_seconds']}s")
print(f"[INFO] output bytes={len(edit_raw)} sha256={edit_sha}")
print("[PASS] EDIT_REST_REQUEST")


## 14. Display source + edited image
## 14. Hiển thị ảnh nguồn + ảnh đã edit

**English**

Display the source and the edited image side by side for immediate visual
comparison, along with the edit instruction and latency.

**Tiếng Việt**

Hiển thị ảnh nguồn và ảnh đã chỉnh sửa cạnh nhau để so sánh trực quan ngay,
kèm câu lệnh edit và latency.


In [ ]:
from IPython.display import HTML
stage("Display source + edited image")
source_b64 = base64.b64encode(source_png_raw).decode("ascii")
edit_b64 = base64.b64encode(edit_raw).decode("ascii")
html = (
    '<div style="display:flex; gap:16px; align-items:flex-start;">'
    '<div><b>Source</b> (' + EDIT_SOURCE.name + ')<br>'
    '<img src="data:image/png;base64,' + source_b64 + '" width="360"></div>'
    '<div><b>Edited</b> (mage-flow-edit-turbo / cuda:1)<br>'
    '<img src="data:image/png;base64,' + edit_b64 + '" width="360"></div>'
    '</div>'
)
display(HTML(html))
print(f"[INFO] instruction={EDIT_PROMPT}")
print(f"[INFO] edit sha256={edit_sha[:16]}... latency={edit['elapsed_seconds']}s")
print("[PASS] SOURCE_EDIT_DISPLAYED")


## 15. Device residency / isolation
## 15. Định vị thiết bị / cô lập GPU

**English**

Verify both workers are still resident (unchanged PIDs during acceptance) and
that T2I reports only `cuda:0` while Edit reports only `cuda:1`. Reads the
orchestrator's PID evidence from `.runtime/acceptance`. No cross-device routing
and no reload happened. The orchestrator's `RUNTIME_EXTRACTION_COUNT=0` is
interpreted as: the orchestrator itself performed **zero additional runtime
extractions after notebook bootstrap**; it is distinct from the notebook's
`BOOTSTRAP_RUNTIME_RESTORE_COUNT`, which may be 1 on a fresh session.

**Tiếng Việt**

Xác minh cả hai worker vẫn thường trú (PID không đổi trong suốt acceptance) và
T2I chỉ báo `cuda:0`, Edit chỉ báo `cuda:1`. Đọc bằng chứng PID từ
`.runtime/acceptance` mà orchestrator ghi lại. Không có routing chéo thiết bị và
không có reload nào xảy ra. `RUNTIME_EXTRACTION_COUNT=0` của orchestrator được
hiểu là: bản thân orchestrator **không thực hiện thêm lần giải nén runtime nào
sau bootstrap của notebook**; con số này khác với `BOOTSTRAP_RUNTIME_RESTORE_COUNT`
của notebook, có thể bằng 1 trong session mới.


In [ ]:
stage("Device residency / isolation")
import json as _json
def _read_json(p):
    p = Path(p)
    return _json.loads(p.read_text()) if p.is_file() else {}
pre = _read_json(ACCEPT_DIR / "pids_before.json")
post = _read_json(ACCEPT_DIR / "pids_after.json")
reveal_path = ACCEPT_DIR / "orchestrator-reveal.txt"
reveal = {}
if reveal_path.is_file():
    for line in reveal_path.read_text().splitlines():
        if "=" in line:
            k, v = line.split("=", 1)
            reveal[k] = v
print(f"[INFO] pids_before={pre}")
print(f"[INFO] pids_after={post}")
print(f"[INFO] orchestrator reveal={reveal}")
print(f"[INFO] ORCHESTRATOR_RUNTIME_EXTRACTION_COUNT={reveal.get('RUNTIME_EXTRACTION_COUNT', 'n/a')}")
info2 = requests.get(f"{REST_URL}/v1/info", headers=api_headers(), timeout=10).json()
if not (info2["t2i"]["device"] == "cuda:0"):
    raise RuntimeError('info2["t2i"]["device"] == "cuda:0"')
if not (info2["edit"]["device"] == "cuda:1"):
    raise RuntimeError('info2["edit"]["device"] == "cuda:1"')
if not (info2["cpu_fallback"] is False):
    raise RuntimeError('info2["cpu_fallback"] is False')
if pre and post:
    if not (pre == post):
        raise RuntimeError(f"PIDs changed during acceptance: {pre} -> {post}")
print("[PASS] DEVICE_ISOLATION_RESIDENT")


## 16. Optional Quick Tunnel
## 16. Quick Tunnel tùy chọn

**English**

Optionally expose the localhost coordinator through a temporary Cloudflare Quick
Tunnel. Only port `8090` is tunnelled; the internal worker ports `8101`/`8102`
are never exposed. The public boundary stays protected by the per-session Bearer
token. This is a demo convenience, not a permanent or SLA-backed endpoint. It
runs only when `ENABLE_QUICK_TUNNEL=1`.

**Tiếng Việt**

Tùy chọn expose coordinator localhost qua Cloudflare Quick Tunnel tạm thời. Chỉ
port `8090` được tunnel; hai port worker nội bộ `8101`/`8102` không bao giờ
được expose. Boundary public vẫn được bảo vệ bằng Bearer token của session. Đây
chỉ là tiện ích demo, không phải endpoint public lâu dài hay có SLA. Chỉ chạy
khi `ENABLE_QUICK_TUNNEL=1`.


In [ ]:
stage("Optional Quick Tunnel")
TUNNEL_URL = None
if os.environ.get("ENABLE_QUICK_TUNNEL") == "1":
    import shutil as _shutil
    if not (_shutil.which("cloudflared")):
        raise RuntimeError("cloudflared not installed; cannot start tunnel")
    tunnel_log = PROJECT_ROOT / ".runtime" / "tunnel.log"
    proc = subprocess.Popen(
        ["cloudflared", "tunnel", "--url", REST_URL, "--no-autoupdate"],
        stdout=open(tunnel_log, "a"), stderr=subprocess.STDOUT)
    print(f"[INFO] cloudflared started pid={proc.pid} log={tunnel_log}")
    deadline = _time.monotonic() + 120
    while _time.monotonic() < deadline:
        text = tunnel_log.read_text() if tunnel_log.is_file() else ""
        import re as _re
        m = _re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", text)
        if m:
            TUNNEL_URL = m.group(0)
            break
        _time.sleep(3)
    if not (TUNNEL_URL):
        raise RuntimeError("tunnel URL not detected within 120s; see .runtime/tunnel.log")
    print(f"[INFO] TUNNEL_URL={TUNNEL_URL}")
    print("[INFO] tunnel exposes coordinator only; worker ports remain localhost-only")
    print("[PASS] QUICK_TUNNEL_STARTED")
else:
    print("[INFO] ENABLE_QUICK_TUNNEL!=1 -> optional tunnel skipped (resident workers untouched)")
    print("[PASS] QUICK_TUNNEL_SKIPPED")


## 17. Optional tunnel smoke tests
## 17. Smoke test qua tunnel (tùy chọn)

**English**

If a Quick Tunnel is running, call `/health`, `/v1/info` and one authenticated
inference through the tunnel using the same request contract and Bearer-token
boundary, while resident workers stay loaded. Restarting or stopping the tunnel
is allowed and never reloads models. If no tunnel is active, this stage records
a skip.

**Tiếng Việt**

Nếu Quick Tunnel đang chạy, gọi `/health`, `/v1/info` và một request inference
có xác thực qua tunnel, giữ nguyên request contract và Bearer-token, trong khi
các worker thường trú vẫn được load. Khởi động lại hay dừng tunnel là được phép
và không bao giờ reload model. Nếu không có tunnel, stage này ghi nhận skip.


In [ ]:
stage("Optional tunnel smoke tests")
if TUNNEL_URL:
    r = requests.get(f"{TUNNEL_URL}/health", timeout=20)
    print(f"[INFO] tunnel GET /health -> {r.status_code} {r.json()}")
    if not (r.status_code == 200):
        raise RuntimeError('r.status_code == 200')
    r = requests.get(f"{TUNNEL_URL}/v1/info", headers=api_headers(), timeout=20)
    if not (r.status_code == 200):
        raise RuntimeError(r.text)
    print(f"[INFO] tunnel /v1/info ok; devices: {r.json()['t2i']['device']} / {r.json()['edit']['device']}")
    print("[PASS] TUNNEL_SMOKE_OK")
else:
    print("[INFO] no active tunnel -> smoke tests skipped")
    print("[PASS] TUNNEL_SMOKE_SKIPPED")


## 18. Final summary
## 18. Tổng kết cuối

**English**

Print a human-readable summary covering hardware, routing, health/readiness,
T2I, Edit, optional tunnel, and known limitations. The public wording is a
production-style demo in a temporary Kaggle session; SLA, HA and multi-tenant
production readiness are not claimed.

**Tiếng Việt**

In tổng kết dễ đọc gồm hardware, routing, health/readiness, T2I, Edit, tunnel
tùy chọn và các giới hạn đã biết. Wording public là bản demo theo phong cách
production trong session Kaggle tạm thời; không tuyên bố SLA, HA hay production
readiness multi-tenant.


In [ ]:
stage("Final production-demo summary")
print("=" * 46)
print("Mage-Flow T4 x2 public REST demo — summary")
print("=" * 46)
print(f"  T2I  : {t2i['model']} on {t2i['device']}  latency={t2i['elapsed_seconds']}s  sha256={t2i_sha}")
print(f"  Edit : {edit['model']} on {edit['device']}  latency={edit['elapsed_seconds']}s  sha256={edit_sha}")
print(f"  source image: {EDIT_SOURCE.name}")
print(f"  cpu_fallback : false (required)")
print(f"  ready        : {requests.get(f'{REST_URL}/ready', headers=api_headers(), timeout=10).json().get('ready')}")
print(f"  tunnel       : {TUNNEL_URL or 'not enabled'}")
print(f"  project source transport : public_github_exact_sha")
print(f"  project source commit    : {EXPECTED_PROJECT_SOURCE_SHA}")
print(f"  project source restore count : {PROJECT_SOURCE_BOOTSTRAP_COUNT}")
print(f"  runtime cache dataset    : dangkhoa2016/mage-flow-t4x2-runtime-cache")
print(f"  runtime archive sha256   : {EXPECTED_RUNTIME_ARCHIVE_SHA256}")
print(f"  bootstrap runtime restore count : {BOOTSTRAP_RUNTIME_RESTORE_COUNT}")
print(f"  orchestrator runtime extraction count : {reveal.get('RUNTIME_EXTRACTION_COUNT', 'n/a')}")
print("  Wording: production-style demo only; temporary Kaggle session;")
print("  not SLA-backed, not HA, not multi-tenant; no CPU fallback.")
print("[PASS] FINAL_SUMMARY")


## 19. Explicit cleanup (optional)
## 19. Cleanup tường minh (tùy chọn)

**English**

Stop the optional tunnel, coordinator, Edit worker and T2I worker exactly once,
at the very end. Cleanup is **gated by `RUN_CLEANUP=1`** so it never runs during
a live acceptance session and never tears down healthy resident workers by
default. The command is idempotent and safe to rerun. It never deletes the
read-only attached Kaggle Models.

**Tiếng Việt**

Dừng tunnel tùy chọn, coordinator, Edit worker và T2I worker đúng một lần, vào
cuối cùng. Cleanup **được gating bởi `RUN_CLEANUP=1`** để không bao giờ chạy
trong lúc acceptance đang sống và mặc định không tắt các worker thường trú khỏe
mạnh. Lệnh này idempotent và an toàn khi chạy lại. Nó không bao giờ xóa Kaggle
Model input read-only.


In [ ]:
stage("Explicit cleanup (optional)")
if os.environ.get("RUN_CLEANUP") == "1":
    if TUNNEL_URL:
        print("[INFO] stopping Quick Tunnel (tunnel process is stopped by owner)")
    run_cmd(["bash", str(PROJECT_ROOT / "scripts" / "stop.sh")], "Stop coordinator/Edit/T2I")
    print("[INFO] attached read-only Kaggle Models left untouched")
    print("[PASS] CLEANUP_DONE")
else:
    print("[INFO] RUN_CLEANUP!=1 -> cleanup skipped; coordinator + model workers remain resident")
    print("[PASS] CLEANUP_SKIPPED")
